# Snowflake Git Integration Setup Guide

This notebook walks through all the steps to create a Git integration in Snowflake, allowing you to sync files from a remote Git repository.

## Prerequisites
- ACCOUNTADMIN role (or appropriate privileges)
- Remote Git repository URL (HTTPS)
- Personal Access Token (PAT) if authentication is required

## Steps Overview
1. Create a Secret (for authentication)
2. Create an API Integration
3. Create the Git Repository
4. Fetch and verify

## Step 1: Create a Secret for Authentication

If your Git repository requires authentication, create a secret with your credentials.

**For GitHub:** Generate a Personal Access Token at https://github.com/settings/tokens

**For Bitbucket:** Use `x-token-auth` as the username

In [ ]:
%%sql -r create_secret
USE ROLE ACCOUNTADMIN;

CREATE OR REPLACE SECRET cicd.public.my_git_secret
  TYPE = PASSWORD
  USERNAME = 'sfc-gh-jleong'           -- Replace with your Git username
  PASSWORD = '';  -- Replace with your PAT

## Step 2: Create an API Integration

The API integration tells Snowflake how to communicate with the Git repository API.

**Options:**
- **No auth:** For public repositories
- **Token auth:** Using the secret created above
- **OAuth (GitHub):** Using Snowflake's built-in GitHub App

In [ ]:
%%sql -r create_api_token
-- Option A: API Integration with token authentication
CREATE OR REPLACE API INTEGRATION my_cicd_git_api_integration
  API_PROVIDER = git_https_api
  API_ALLOWED_PREFIXES = ('https://github.com/sfc-gh-jleong/cicd-example')  -- Replace with your repo prefix
  ALLOWED_AUTHENTICATION_SECRETS = (cicd.public.my_git_secret)
  ENABLED = TRUE;

## Step 3: Grant Privileges and Create the Git Repository

Grant the necessary privileges, then create the Git repository clone.

In [ ]:
%%sql -r grant_privs
-- Grant privileges to a role (if not using ACCOUNTADMIN directly)
GRANT CREATE GIT REPOSITORY ON SCHEMA CICD.ETL TO ROLE SYSADMIN;
GRANT USAGE ON INTEGRATION my_git_api_integration TO ROLE SYSADMIN;
GRANT USAGE ON SECRET my_git_secret TO ROLE SYSADMIN;

In [ ]:
%%sql -r create_repo
-- Create the Git repository clone
CREATE OR REPLACE GIT REPOSITORY CICD.ETL.my_git_repo
  API_INTEGRATION = my_cicd_git_api_integration
  GIT_CREDENTIALS = cicd.public.my_git_secret
  ORIGIN = 'https://github.com/sfc-gh-jleong/cicd-example';  -- Replace with your repo URL

## Step 4: Fetch and Verify

Fetch the latest from the remote repository and explore the contents.

In [ ]:
%%sql -r fetch_repo
-- Fetch latest changes from remote
ALTER GIT REPOSITORY CICD.ETL.my_git_repo FETCH;

In [ ]:
%%sql -r show_repos
-- Show all Git repositories in the account
SHOW GIT REPOSITORIES;

In [ ]:
%%sql -r show_branches
-- List branches in the repository
SHOW GIT BRANCHES IN GIT REPOSITORY CICD.ETL.my_git_repo;

In [ ]:
%%sql -r show_tags
-- List tags in the repository
SHOW GIT TAGS IN GIT REPOSITORY CICD.ETL.my_git_repo;

In [ ]:
%%sql -r list_files
-- List files in the main branch
LS @CICD.ETL.my_git_repo/branches/main/;

## Using Git Repository Files

Once set up, you can reference files from the repository in your code:

```sql
-- Execute a SQL script from the repo
EXECUTE IMMEDIATE FROM @my_git_repo/branches/main/scripts/setup.sql;

-- Create a UDF with handler code from the repo  
CREATE FUNCTION my_func()
  RETURNS STRING
  LANGUAGE PYTHON
  IMPORTS = ('@my_git_repo/branches/main/src/utils.py')
  HANDLER = 'utils.main';
```

## Cleanup (Optional)

In [ ]:
%%sql -r cleanup
-- Drop the Git repository
-- DROP GIT REPOSITORY CICD.ETL.my_git_repo;

-- Drop the API integration
-- DROP API INTEGRATION my_git_api_integration;

-- Drop the secret
-- DROP SECRET my_git_secret;